# Revision analyses R2 — feature ablation (Reviewer 2, question 2)
Tests whether the 21-kyr sin/cos features and `interp_age` improve prediction, and what happens when the 21-kyr period is replaced by 19, 23, 41 or 100 kyr. This is the long-running notebook (several hours). Every fold is checkpointed to Drive, so re-running all cells resumes where it stopped.


## 0 · Setup
Mounts Drive, defines paths, constants and the helpers used throughout. Run all cells in order.
Everything is written to `artifacts/revision_2026_11/` on Drive; nothing in the original run folders is modified.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import subprocess, sys, importlib, importlib.util
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split("==")[0].split(">=")[0].replace("-", "_")
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
_pip("catboost", "xgboost")

In [ ]:
import os, json, time, warnings, zlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

# ---- Paths -----------------------------------------------------------------------
_candidates = [Path(os.environ["SISAL_BASE"])] if "SISAL_BASE" in os.environ else []
_candidates += [Path("/content/drive/MyDrive/SISALv3_Speleothem"),
                Path("/content/drive/My Drive/SISALv3_Speleothem")]
base = next((p for p in _candidates if p.exists()), _candidates[0])
ARTIFACTS   = base / "artifacts"
LOCO_DIR    = ARTIFACTS / "20260616T224121Z"
WITHIN_DIR  = ARTIFACTS / "20260612T233845Z"
WITHIN_PRED = WITHIN_DIR / "predictions_within"
MODELS_DIR  = next((p for p in [base / "webapp/models", base / "models"] if p.exists()), base / "webapp/models")
RAW_SISAL   = base / "data/sisalv3_database_mysql_csv/sisalv3_database_mysql_csv/sisalv3_csv"
OUT         = ARTIFACTS / "revision_2026_11"
FIG         = OUT / "figures"
for d in (OUT, FIG): d.mkdir(parents=True, exist_ok=True)
print("base       :", base, "| exists:", base.exists())
print("models     :", MODELS_DIR, "| exists:", MODELS_DIR.exists())
print("raw SISAL  :", RAW_SISAL, "| exists:", RAW_SISAL.exists())
print("output dir :", OUT)

# ---- Constants (match the manuscript) --------------------------------------------
TARGETS  = ["d18O_measurement", "d13C_measurement", "Mg_Ca_measurement", "Sr_Ca_measurement"]
T_LBL    = {"d18O_measurement": "d18O", "d13C_measurement": "d13C",
            "Mg_Ca_measurement": "Mg/Ca", "Sr_Ca_measurement": "Sr/Ca"}
T_UNIT   = {"d18O_measurement": "permil", "d13C_measurement": "permil",
            "Mg_Ca_measurement": "mmol/mol", "Sr_Ca_measurement": "mmol/mol"}
MODELS   = ["RandomForest", "XGBoost", "CatBoost"]
LOCO_BEST  = {"d18O_measurement": "RandomForest", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "RandomForest"}   # Table 1
WCBG_BEST  = {"d18O_measurement": "CatBoost", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "XGBoost"}        # Table 2
PUB_LOCO_R2 = {"d18O_measurement": 0.516, "d13C_measurement": 0.182,
               "Mg_Ca_measurement": -0.553, "Sr_Ca_measurement": 0.564}               # LOCO_BEST rows
PUB_WCBG_R2 = {("RandomForest", "d18O_measurement"): 0.971, ("XGBoost", "d18O_measurement"): 0.974,
               ("CatBoost", "d18O_measurement"): 0.977, ("RandomForest", "d13C_measurement"): 0.913,
               ("XGBoost", "d13C_measurement"): 0.906, ("CatBoost", "d13C_measurement"): 0.917,
               ("RandomForest", "Mg_Ca_measurement"): 0.925, ("XGBoost", "Mg_Ca_measurement"): 0.941,
               ("CatBoost", "Mg_Ca_measurement"): 0.958, ("RandomForest", "Sr_Ca_measurement"): 0.981,
               ("XGBoost", "Sr_Ca_measurement"): 0.987, ("CatBoost", "Sr_Ca_measurement"): 0.985}
PUB_LI_RMSE = {"d18O_measurement": 0.469, "d13C_measurement": 0.897,
               "Mg_Ca_measurement": 1.866, "Sr_Ca_measurement": 0.072}                # Table 2, LI rows
NUM_FEATS = ["latitude", "longitude", "elevation", "depth_sample", "interp_age",
             "P_Ca_measurement", "U_Ca_measurement", "Ba_Ca_measurement",
             "age_sin", "age_cos", "log_depth_sample", "precip_d18o_bw02", "abs_latitude",
             *[f"wc_bio{i}" for i in range(1, 20)]]                                  # pipeline CONFIG
NOM_FEATS = ["mineralogy", "monitoring", "hemisphere", "lat_band"]
SEED = 42
REPORT = {}   # everything the final report cell prints

# ---- CorrelationDropper (must exist before any joblib.load) ------------------------
class CorrelationDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = float(threshold)
    def fit(self, X, y=None):
        Xa = np.asarray(X, dtype=float)
        self.n_features_in_ = Xa.shape[1]
        if Xa.shape[0] < 4 or Xa.shape[1] < 2:
            self.kept_mask_ = np.ones(self.n_features_in_, dtype=bool)
            self.dropped_indices_ = np.array([], dtype=int); return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.corrcoef(Xa, rowvar=False)
        corr = np.where(np.isfinite(corr), corr, 0.0)
        upper = np.triu(np.abs(corr), k=1)
        to_drop = set()
        for i in range(corr.shape[0]):
            for j in range(i + 1, corr.shape[1]):
                if upper[i, j] >= self.threshold and j not in to_drop:
                    to_drop.add(j)
        self.kept_mask_ = np.array([k not in to_drop for k in range(self.n_features_in_)], dtype=bool)
        self.dropped_indices_ = np.array(sorted(to_drop), dtype=int)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.kept_mask_]
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = np.array([f"x{i}" for i in range(self.n_features_in_)])
        return np.asarray(input_features)[self.kept_mask_]

def make_preprocessor(num_feats, nom_feats=NOM_FEATS):
    """Identical to the pipeline's make_column_transformer_factory."""
    num = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", RobustScaler()),
                    ("drop_corr", CorrelationDropper(0.95))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2,
                                          sparse_output=False))])
    return ColumnTransformer([("num", num, list(num_feats)), ("cat", cat, list(nom_feats))],
                             remainder="drop", verbose_feature_names_out=False)

def rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); return float(np.sqrt(np.mean((a - b) ** 2)))
def r2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ss = np.sum((a - a.mean()) ** 2); return float(1 - np.sum((a - b) ** 2) / ss) if ss > 0 else np.nan

# ---- Data ------------------------------------------------------------------------
master = pd.read_parquet(ARTIFACTS / "_data" / "master_table_enriched.parquet").reset_index(drop=True)
master["_pos"] = master.groupby("entity_id").cumcount()          # row order inside each entity
_mono = master.groupby("entity_id")["interp_age"].apply(lambda a: bool((a.diff().dropna() >= 0).all()))
print(f"[OK] master_table {master.shape}; entities with age-monotonic row order: {_mono.sum()}/{len(_mono)}")

within_preds = {}
for p in WITHIN_PRED.glob("*.parquet"):
    name, tgt = p.stem.split("_", 1)
    within_preds[(name, tgt)] = pd.read_parquet(p)
print(f"[OK] within-cave prediction files: {len(within_preds)}")

prod = {}
for m in MODELS:
    for t in TARGETS:
        f = MODELS_DIR / f"{m}_{t}.joblib"
        if f.exists():
            prod[(m, t)] = joblib.load(f)
print(f"[OK] production pipelines loaded: {len(prod)}")
def kept_numeric_features(pipe):
    ct = pipe.named_steps["preprocess"]
    cols = [c for n, tr, cs in ct.transformers_ if n == "num" for c in cs]
    dc = ct.named_transformers_["num"].named_steps.get("drop_corr")
    return [c for c, k in zip(cols, dc.kept_mask_) if k] if dc is not None else cols
_feat_check = {k: ("interp_age" in kept_numeric_features(v)) for k, v in prod.items()}
REPORT["interp_age_is_predictor_in_production_models"] = f"{sum(_feat_check.values())}/{len(_feat_check)}"
print("interp_age used as predictor in production models:", REPORT["interp_age_is_predictor_in_production_models"])

## 1 · Configuration
**What this tests (R2-Q2).** The published models use `interp_age` *and* the 21-kyr sin/cos pair. Variants:

| Tier | Variant | Age information given to the model |
|---|---|---|
| 1 | `full` | interp_age + 21-kyr sin/cos (as published) |
| 1 | `no_harmonics` | interp_age only |
| 1 | `no_interp_age` | 21-kyr sin/cos only (what §2 of the manuscript describes) |
| 1 | `no_age` | none |
| 2 | `p19`, `p23`, `p41`, `p100` | `HARMONIC_BASE` variant with the period replaced (kyr) |

Each variant is refitted with the **best model per proxy and its tuned hyper-parameters** (cloned from the production
pipelines) under LOCO (41 folds) and WCBG (the same 46 saved blocks). Every fold is cached on Drive, so after a
disconnect simply re-run all cells and it resumes. Two Colab sessions can run in parallel on different
`TARGETS_TO_RUN` (e.g. the RF proxies in one, the CatBoost proxies in another).

Rough cost on Colab Pro: RF proxies ~5–25 s per fold; CatBoost (depth 10, 900 iterations) ~40–60 s per fold on CPU,
~10–15 s on GPU. Tier 1 on both protocols is the priority; Tier 2 defaults to LOCO only.

In [ ]:
TARGETS_TO_RUN   = TARGETS                       # or a subset, e.g. ["d18O_measurement", "Sr_Ca_measurement"]
ABLATION_MODEL   = dict(LOCO_BEST)               # RF for d18O and Sr/Ca, CatBoost for d13C and Mg/Ca
TIER1            = ["full", "no_harmonics", "no_interp_age", "no_age"]
TIER2            = ["p19", "p23", "p41", "p100"]
HARMONIC_BASE    = "full"                        # Tier-2 periods replace 21 kyr inside this variant
PROTOCOLS        = {1: ["LOCO", "WCBG"], 2: ["LOCO"]}
RUN_TIERS        = [1, 2]
CATBOOST_GPU     = "auto"                        # "auto" | True | False
MAX_FOLDS        = None                          # e.g. 2 for a dry run

ABL = OUT / "ablation"; ABL.mkdir(exist_ok=True)

def variant_spec(v):
    if v == "full":          return [c for c in NUM_FEATS], 21.0
    if v == "no_harmonics":  return [c for c in NUM_FEATS if c not in ("age_sin", "age_cos")], 21.0
    if v == "no_interp_age": return [c for c in NUM_FEATS if c != "interp_age"], 21.0
    if v == "no_age":        return [c for c in NUM_FEATS if c not in ("age_sin", "age_cos", "interp_age")], 21.0
    if v.startswith("p"):    return variant_spec(HARMONIC_BASE)[0], float(v[1:])
    raise ValueError(v)

def features_for(period_kyr):
    X = master.drop(columns=TARGETS + ["_pos"]).copy()
    X["age_sin"] = np.sin(2 * np.pi * X["interp_age"].astype(float) / (period_kyr * 1000))
    X["age_cos"] = np.cos(2 * np.pi * X["interp_age"].astype(float) / (period_kyr * 1000))
    return X

def _gpu_available():
    try: return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception: return False
USE_GPU = _gpu_available() if CATBOOST_GPU == "auto" else bool(CATBOOST_GPU)
print("CatBoost on GPU:", USE_GPU)

def make_pipe(model, tgt, num_feats):
    est = clone(prod[(model, tgt)].named_steps["estimator"])
    if model == "CatBoost":
        est.set_params(task_type="GPU" if USE_GPU else "CPU", allow_writing_files=False, verbose=False)
    return Pipeline([("preprocess", make_preprocessor(num_feats)), ("estimator", est)])

# sanity: rebuilt full-variant preprocessor uses the same inputs as production
for t in TARGETS:
    m = ABLATION_MODEL[t]
    assert set(variant_spec("full")[0]) == set(c for n, tr, cs in prod[(m, t)].named_steps["preprocess"].transformers_
                                               if n == "num" for c in cs), f"feature mismatch for {m} {t}"
print("[OK] full variant = production feature set")

# WCBG blocks: exactly the saved ones, per (model, target)
WCBG_BLOCKS = {}
for t in TARGETS:
    pf = within_preds[(ABLATION_MODEL[t], t)]
    WCBG_BLOCKS[t] = {f: master.index[master.sample_id.isin(g.sample_id)].to_numpy() for f, g in pf.groupby("fold")}
LOCO_FOLDS = {s: master.index[master.site_id == s].to_numpy() for s in sorted(master.site_id.unique())}
print(f"[OK] WCBG folds per target: {[len(v) for v in WCBG_BLOCKS.values()]}; LOCO folds: {len(LOCO_FOLDS)}")

## 2 · Run (resumable)

In [ ]:
def run_combo(variant, tgt, protocol):
    model = ABLATION_MODEL[tgt]; feats, period = variant_spec(variant)
    cache = ABL / f"{protocol}_{variant}_{model}_{tgt}.parquet"
    done = pd.read_parquet(cache) if cache.exists() else pd.DataFrame()
    folds = LOCO_FOLDS if protocol == "LOCO" else WCBG_BLOCKS[tgt]
    keys = list(folds)[:MAX_FOLDS] if MAX_FOLDS else list(folds)
    todo = [k for k in keys if done.empty or k not in set(done.fold)]
    if not todo: return done
    X = features_for(period); y = master[tgt].to_numpy(float); parts = [done] if len(done) else []
    t0 = time.time()
    for j, k in enumerate(todo):
        te = folds[k]; tr = np.setdiff1d(np.arange(len(master)), te)
        pipe = make_pipe(model, tgt, feats).fit(X.iloc[tr], y[tr])
        parts.append(pd.DataFrame(dict(fold=k, sample_id=master.loc[te, "sample_id"].to_numpy(),
                                       y_true=y[te], y_pred=pipe.predict(X.iloc[te]))))
        pd.concat(parts, ignore_index=True).to_parquet(cache, index=False)        # checkpoint every fold
        if j == 0 or (j + 1) % 10 == 0 or j == len(todo) - 1:
            el = (time.time() - t0) / (j + 1)
            print(f"  {protocol:4s} {variant:13s} {T_LBL[tgt]:5s} fold {j+1}/{len(todo)}  {el:.0f}s/fold  "
                  f"~{el*(len(todo)-j-1)/60:.0f} min left")
    return pd.concat(parts, ignore_index=True)

RESULTS = {}
for tier in RUN_TIERS:
    variants = TIER1 if tier == 1 else TIER2
    for protocol in PROTOCOLS[tier]:
        for t in TARGETS_TO_RUN:
            for v in variants:
                RESULTS[(protocol, v, t)] = run_combo(v, t, protocol)
print("[OK] all requested combinations complete")

## 3 · Summarise
Pooled out-of-fold R² and RMSE per variant, the change versus `full`, and a paired Wilcoxon test on per-fold RMSE
(variant vs `full`). The `full` row should reproduce the published pooled R² (small differences are expected for
CatBoost on GPU).

In [ ]:
rows = []
for (protocol, v, t), df in RESULTS.items():
    if df.empty: continue
    base_df = RESULTS.get((protocol, "full", t))
    d = dict(protocol=protocol, target=T_LBL[t], model=ABLATION_MODEL[t], variant=v,
             n_folds=df.fold.nunique(), pooled_r2=round(r2(df.y_true, df.y_pred), 3),
             pooled_rmse=round(rmse(df.y_true, df.y_pred), 4))
    if base_df is not None and len(base_df) and v != "full":
        common = sorted(set(df.fold) & set(base_df.fold))
        a = base_df[base_df.fold.isin(common)]; b = df[df.fold.isin(common)]
        d["delta_r2_vs_full"] = round(r2(b.y_true, b.y_pred) - r2(a.y_true, a.y_pred), 3)
        fa = a.groupby("fold").apply(lambda g: rmse(g.y_true, g.y_pred))
        fb = b.groupby("fold").apply(lambda g: rmse(g.y_true, g.y_pred)).reindex(fa.index)
        d["folds_worse_than_full"] = f"{int((fb > fa).sum())}/{len(fa)}"
        d["wilcoxon_p_fold_rmse"] = round(stats.wilcoxon(fa, fb).pvalue, 4) if len(fa) >= 6 and (fa != fb).any() else None
    if v == "full":
        d["published_r2"] = PUB_LOCO_R2[t] if protocol == "LOCO" else PUB_WCBG_R2.get((ABLATION_MODEL[t], t))
    rows.append(d)
order = {v: i for i, v in enumerate(TIER1 + TIER2)}
SUMMARY = pd.DataFrame(rows).sort_values(["protocol", "target", "variant"], key=lambda s: s.map(order) if s.name == "variant" else s)
SUMMARY.to_csv(OUT / "ablation_summary.csv", index=False)
REPORT["ablation_summary"] = SUMMARY
print(SUMMARY.to_string(index=False))

# Importance of the age features in the full LOCO-best production models (impurity / CatBoost importance)
imp_rows = []
for t in TARGETS:
    pipe = prod[(ABLATION_MODEL[t], t)]; ct = pipe.named_steps["preprocess"]
    kept = kept_numeric_features(pipe)
    n_cat = ct.named_transformers_["cat"].named_steps["ohe"].transform(
        ct.named_transformers_["cat"].named_steps["impute"].transform(master[NOM_FEATS].head(1))).shape[1]
    names = kept + [f"cat_{i}" for i in range(n_cat)]
    est = pipe.named_steps["estimator"]
    imp = np.asarray(est.get_feature_importance() if ABLATION_MODEL[t] == "CatBoost" else est.feature_importances_, float)
    imp = imp / imp.sum(); s = pd.Series(imp, index=names[:len(imp)]).sort_values(ascending=False)
    imp_rows.append(dict(target=T_LBL[t], model=ABLATION_MODEL[t],
                         rank_interp_age=int(list(s.index).index("interp_age") + 1) if "interp_age" in s else None,
                         share_interp_age=round(float(s.get("interp_age", 0)), 3),
                         share_age_sin_cos=round(float(s.get("age_sin", 0) + s.get("age_cos", 0)), 3),
                         top5=", ".join(s.index[:5])))
REPORT["production_model_age_feature_importance"] = pd.DataFrame(imp_rows)
print(REPORT["production_model_age_feature_importance"].to_string(index=False))

## 4 · Report — summary of all results

In [ ]:
_lines = ["=" * 100, "SISAL REVISION — FEATURE ABLATION REPORT (R2)", f"generated: {pd.Timestamp.now(tz='UTC')}",
          f"targets run: {[T_LBL[t] for t in TARGETS_TO_RUN]} | tiers: {RUN_TIERS} | protocols: {PROTOCOLS} | "
          f"CatBoost GPU: {USE_GPU} | MAX_FOLDS: {MAX_FOLDS}",
          f"interp_age used by production models: {REPORT['interp_age_is_predictor_in_production_models']}", "=" * 100]
for k in ["ablation_summary", "production_model_age_feature_importance"]:
    _lines += ["", f"### {k}", REPORT[k].to_string(index=False)]
_txt = "\n".join(_lines); (OUT / "report_nb2.txt").write_text(_txt); print(_txt)